<center>МИНИСТЕРСТВО НАУКИ И ВЫСШЕГО ОБРАЗОВАНИЯ РОССИЙСКОЙ ФЕДЕРАЦИИ </center>
<center>ФЕДЕРАЛЬНОЕ ГОСУДАРСТВЕННОЕ БЮДЖЕТНОЕ ОБРАЗОВАТЕЛЬНОЕ УЧРЕЖДЕНИЕ ВЫСШЕГО ОБРАЗОВАНИЯ </center>
<center>«НОВОСИБИРСКИЙ ГОСУДАРСТВЕННЫЙ ТЕХНИЧЕСКИЙ УНИВЕРСИТЕТ»</center>
<center>Кафедра Вычислительной техники </center>
<br>
<center> <b> <font size="5">  ОТЧЁТ </font>  </b>  </center>   
<center><font size="3">по лабораторной работе №3</font></center>
<center><font size="3">по дисциплине: «Системы искусственного интеллекта и машинное обучение» </font></center>
<br>

Выполнили:
- Болотенко Н.А.
- Левицкий А.В.

Проверил: Пронюшкина А.Н.

<center>  Новосибирск, 2025  </center>


## Цель работы

Знакомство и работа с моделью машинного обучения типа "многослойный перцептрон" для решения задачи регрессии с использованием библиотеки Tensorflow.

## Основная часть
---

### Подключение пакетов

Перед началом работы нужно убедиться, что необходимые для работы пакеты установлены в системе.

In [ ]:
import sys
print(f"Версия Python - {sys.version}")
print(f"Путь к интерпретатору Python - {sys.executable}")

In [ ]:
import pandas as pd
import numpy  as np

import sklearn
from sklearn import linear_model
from sklearn import ensemble
from sklearn import metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from matplotlib import pyplot as plt
from matplotlib import cm
import seaborn as sns

import tensorflow as tf
from IPython.display import clear_output
import plotly.graph_objects as go

### Объявление функций 

In [ ]:
def plot_difference(y_test, y_pred) -> None:
    '''
    Функция построения графиков
    :param y_test: - проверочные значения целевой переменной
    :param y_pred: - вычисленные значения целевой переменной
    '''
    plt.figure(figsize=(12,6))
    
    plt.subplot(121)
    plt.scatter(y_test, y_pred,  alpha=0.1, color = "#17becf")
    plt.axline((0, 0), slope=1, color='black', linestyle='--', linewidth=3, alpha=0.7,)
    range_test, range_pred = np.max(y_test)- np.min(y_test) , np.max(y_pred)- np.min(y_pred)
    if range_test/range_pred <4 and range_test/range_pred > 0.25:
        plt.gca().set_aspect('equal')
        axmin, axmax = np.min([y_test, y_pred]), np.max([y_test, y_pred])
        plt.xlim([axmin, axmax]); plt.ylim([axmin, axmax]);
    plt.title('Диаграмма рассеяния вычисленных значений');
    plt.xlabel('Проверочное Y')
    plt.ylabel('Вычисленное Y')
    plt.grid(True)

    plt.subplot(122) # 1 row, 2 column, 2 index on grid
    plt.scatter(y_test, (y_test - y_pred)**2,  alpha=0.1, color = "#17becf")
    plt.title('Диаграмма рассеяния квадрата абсолютной ошибки')
    plt.xlabel('Проверочное Y')
    plt.ylabel('Квадрат абсолютной ошибки')
    plt.grid(True)

In [ ]:
def stats(y_test, y_pred):
    '''
    Вычисление и вывод метрик: MAE, RMSE, R2. Используются функции из библиотеки sklearn
    На основе сравнения проверочных и вычисленных.
    :param y_test: - проверочные значения целевой переменной
    :param y_pred: - вычисленные значения целевой переменной
    '''
    mae  = metrics.mean_absolute_error(y_test, y_pred)
    mse  = metrics.mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = metrics.r2_score(y_test, y_pred)
    print ("MAE : {:>9,.3f} (средняя абсолютная ошибка)".format(mae))
    print ("MSE : {:>9,.6f} (среднеквадратичная ошибка)".format(mse))
    print ("RMSE: {:>9,.6f} (кв. корень из среднеквадратичной ошибки)".format(rmse))
    print ("R2  : {:>9,.3f} (коэфф. детерминации)".format(r2))
    return {"MAE":mae, "MSE":mse, "RMSE":rmse, "R2":r2}

In [ ]:
def my3dplot (  x_train: pd.DataFrame, y_train: pd.DataFrame,  isDrawTrain: bool,
                x_test:  pd.DataFrame, y_test: pd.DataFrame,    isDrawTest: bool,
                y_pred:  pd.DataFrame,            isDrawPred: bool,
                x1Name: str, x2Name: str, y_targetName: str,
                pointSize = 5,
                pointTransparency  = 0.25,
                meshTransparency   = 0.5
                ) -> None:
    """
    Отображает 3D график исходных и вычисленых точек
    x_test, y_test, x_train, y_train - обучающая и проверочная части исходной таблицы  (тип DataFrame)
    y_pred, - вычисленные значения (тип DataFrame)
    x1Name, x2Name - имя столбца для x1 и x2 (тип строковый)
    y_targetName   - имя столбца целевого    (тип строковый)
    """
    trace_ModelPredicted = go.Mesh3d(
                       x= x_test[ x1Name ].values,
                       y= x_test[ x2Name ].values,
                       z= y_pred.reshape(-1), # <-- Вычисленные
                       name="Вычисленный",
                       opacity=meshTransparency,
                       #alphahull=1,
                       color='rgb(244,100,100)',
                      )

    trace_Train = go.Scatter3d (x= x_train[ x1Name ].values,
                                y= x_train[ x2Name ].values,
                                z= y_train.values,
                                name="Обучающие",
                                mode='markers',
                                marker=dict(
                                    size=pointSize,
                                    color='#1f77b4',
                                    opacity=pointTransparency
                                ))

    trace_Test = go.Scatter3d ( x= x_test[ x1Name ].values,
                                y= x_test[ x2Name ].values,
                                z= y_test.values,
                                name="Проверочные",
                                mode='markers',
                                marker=dict(
                                    size=pointSize,
                                    color='#ff7f0e',
                                    opacity=pointTransparency
                               ))

    ListForDraw = []
    if isDrawTrain: ListForDraw.append(trace_Train)
    if isDrawTest:  ListForDraw.append(trace_Test)
    if isDrawPred:  ListForDraw.append(trace_ModelPredicted)

    fig = go.Figure( data=ListForDraw)

    fig.update_layout(
        width=800,
        height=600,
        title='Завиcимость {} от ({}, {})'.format(y_targetName, x1Name, x2Name),
        scene=dict(
            xaxis_title=x1Name,
            yaxis_title=x2Name,
            zaxis_title=y_targetName,
        ),
    )
    fig.show()

In [ ]:
def printModelWeights(model):
  for layer in model.layers:
      print(f'{"#"*100}')
      print(f"#### Имя слоя:{layer.name};  Тип слоя: {layer.__class__} ##########", end='')
      print(f"\nВид ф-ии активации слоя: {layer.get_config()['activation']}", end='')
      print(f"\nКол-во ВХодов  слоя: {layer.input.shape[1]}", end='')
      print(f"\nКол-во ВЫХодов слоя: {layer.output.shape[1]}", end='')
      if type(layer) is tf.keras.layers.Dense:
          print(f"\nКол-во нейронов слоя:    {layer.get_config()['units']}", end='')
          print(f"\n\nВесовые коэфф. weight_i_j=\n{layer.weights[0].numpy()}")
          print(f"\nВесовые коэфф. bias_i_j=\n{layer.bias.numpy()}")
      elif type(layer) is tf.keras.layers.Activation:
          pass
      print('\n')

### Tensorflow. Проверка работы нейросети на искусственных данных

Для последующего сравнения получим модель нейронной сети обученной на почти "идеальных" входных данных, т.е. с четко прослеживаемой зависимостью переменных и контролируемым уровнем шумов.

Цель данного шага - продемонстрировать работоспособность применения НС для решения задачи регрессии.

#### Набор данных для демонстрации

Создадим несколько искусственных наборов данных с явной зависимостью ЕДИНСТВЕННОГО y от ЕДИНСТВЕННОГО x.

После создания преобразуем данные к типу pandas.DataFrame, т.к. загруженные данные по варианту будут иметь указанный тип.

Создадим переменную X как набор значений области определения функции в диапазоне [-1.0, 1.0].

In [ ]:
N = 1000
X = np.linspace(-1., 1., N).reshape(-1, 1)

Получим набор значения для имитации случайных отклонений для зависимой величины `y`.

In [ ]:
noiseLevel= 0.05
Y_noise = np.random.normal(-noiseLevel,noiseLevel,N).reshape(-1, 1)

Используя функции различного вида, получим значения `y` для каждого `x`.

In [ ]:
transparencyVal = 0.3  # Прозрачность точек графика
markerSize = 10        # Размер точки графика

Обучим НС для объяснения наложения циклических функций

In [ ]:
# Наложение циклических функций
Y_hard = np.cos(X*10)  - 1*np.sin(20*X) - 1*X**3
Y_hard = Y_hard+Y_noise*2
plt.scatter(X, Y_hard, s=markerSize, color="black", alpha=transparencyVal)

Но сначала сделаем из данных столбец - будут отдельные записи.

In [ ]:
x_train = X.copy().reshape(-1,1)
y_train = Y_hard.copy().reshape(-1,1)

Нормализуем исходные данные для приведения всех значений к единой шкале.

In [ ]:
NormalizerX_EXAMPLE = MinMaxScaler().fit(x_train)
NormalizerY_EXAMPLE = MinMaxScaler().fit(y_train)
xNorm_train = NormalizerX_EXAMPLE.transform(x_train)
yNorm_train = NormalizerY_EXAMPLE.transform(y_train)

plt.scatter(xNorm_train, yNorm_train, s=markerSize, color="black", alpha=transparencyVal)
plt.title("Выбранный нормализованный набор")
plt.ylabel("Целевой Y"); plt.xlabel("Объясняющий X");
plt.grid(True, alpha=0.5)  # Сетка.

Структура НС - с использованием [полносвязных слоев](https://adgefficiency.com/guide-deep-learning/#:~:text=The%20fully%20connected%20layer%20is,shape%20of%20the%20output%20layer.), позволяющих объяснять [любую функцию](https://adgefficiency.com/guide-deep-learning/#:~:text=This%20lack%20of%20structure%20is%20what%20gives%20neural%20networks%20of%20fully%20connected%20layers%20(of%20sufficient%20depth%20%26%20width)%20the%20ability%20to%20approximate%20any%20function%20%2D%20known%20as%20the%20Universal%20Approximation%20Theorem.) (см. теорему Цыбенко)

Наша функция сложная - понадобится несколько слоев. Можем использовать следующие [фукнции активации](https://www.tensorflow.org/api_docs/python/tf/keras/activations): *RELU*, зарекомендовавший себя в роли [простого алгоритмически и эффективного](https://towardsdatascience.com/activation-functions-in-neural-networks-how-to-choose-the-right-one-cb20414c04e5/#:~:text=In%20the%20hidden%20layers%2C%20the%20activation%20function%20ReLU%20and%20its%20variants%20have%20established%20themselves%20as%20they%20are%20very%20efficient%20to%20calculate%20and%20at%20the%20same%20time%20avoid%20the%20vanishing%20gradient%20problem%2C%20which%20is%20an%20important%20factor%20in%20this%20area%20of%20architecture.), а также *tanh*, [объясняющего сложные зависимости](https://towardsdatascience.com/activation-functions-in-neural-networks-how-to-choose-the-right-one-cb20414c04e5/#:~:text=It%20is%20also%20a%20non%2Dlinear%20activation%20function%2C%20which%20enables%20the%20model%20to%20learn%20more%20complex%20relationships).

Зададим 3(4) промежуточных слоя, последний будет с гиперболической функцией активации. [Типов слоев](https://www.tensorflow.org/api_docs/python/tf/keras/layers) много, мы же будем использовать обычные плотные слои и отдельную функцию активации.

In [ ]:
totalHistoryLossTrain=[] # Вспомогательный список для хранения полной истории обучения
totalHistoryLossTest=[]  # Вспомогательный список для хранения полной истории обучения
globalEpochCounter = 1   # Счётчик выполненных эпох обучения НС

model = tf.keras.models.Sequential()
model.add( tf.keras.layers.Input(shape=(  1 ,))) # Входной слой
# Скрытые слои
# "units" - кол-во узлов у слоя
# "activation" - функция активации
model.add(tf.keras.layers.Dense(units= 500 ,  activation=None))
model.add(tf.keras.layers.Activation( activation=tf.keras.activations.relu))
model.add(tf.keras.layers.Dense(units= 250 ,  activation=tf.keras.activations.relu))
model.add(tf.keras.layers.Dense(units= 50 ,  activation=tf.keras.activations.tanh))
model.add(tf.keras.layers.Dense(units=1,  activation=None)) # Выходной полносвязный слой, Без нелинейной функции активации

fLoss=tf.keras.losses.MeanSquaredError()
fOptimizer=tf.keras.optimizers.Adam(learning_rate=0.01)
fMetricList  = ['accuracy']

model.compile(
    loss      =fLoss, # Выбор функции оценки ошибок/потерь
    optimizer =fOptimizer, # Выбор функции для минимзации значения выбранной оценки ошибок/потерь
    metrics   = fMetricList # Дополнительные метрики для оценки ошибок/потерь
)

print(model.summary())

[Функция ошибок](https://www.tensorflow.org/api_docs/python/tf/keras/losses) позволяет оценивать то, насколько хорошо(или плохо) модель объясняет **тестовые** данные в *процессе обучения*.

In [ ]:
# Выполнить заданное кол-во эпох обучения
epochForTrain = 500 # Кол-во эпох обучения
for currEpochNum in range(1, epochForTrain+1):
    # 3) Цикл обучения/продолжения обучения сети

    # Вызов метода для выполнения обучения (настройки коэфф. нейросети)
    history = model.fit(
        xNorm_train,  # Обучающие X
        yNorm_train,  # Обучающие Y
        #validation_data=(xNorm_test, yNorm_test), # Опционально проверочные X и Y, для вычисления оценок потерь (loss)
        epochs=1,       # Кол-во эпох обучения
        batch_size=100, # Внутри каждой эпохи, проводить обучение пакетами/порциями по batch_size строк
        verbose=1,
    )

    #  Каждые N эпох выводить график текущей функции НС. Только в условиях ЕДИНСТВЕННОГО x.
    if xNorm_train.shape[1] == 1:
        if (currEpochNum) % 3 == 0:
            yNorm_pred = model.predict(xNorm_train, verbose=0) # Вычислить ответы НС
            clear_output(wait=True) # Очистить окно вывода
            print(f'currEpochNum: {currEpochNum}/{epochForTrain}')
            plt.figure(figsize=(6, 4))
            plt.scatter(xNorm_train, yNorm_train,  s=markerSize, color='black', alpha=transparencyVal, label =  'Исходные данные')
            plt.scatter(xNorm_train, yNorm_pred,   s=markerSize, color='red',   alpha=transparencyVal, label = f'График функции нейронной сети. MSE {history.history["loss"][-1]:>9.6f}')
            plt.title(f'Выбранный нормализованный набор. ({globalEpochCounter:>4} эпоха)')
            plt.ylabel('Целевой Y'); plt.xlabel('Объясняющий X');
            plt.ylim( (-0.1, 1.1))
            plt.legend(loc='lower right')
            plt.grid(True, alpha=0.3)
            plt.show()
            plt.pause(0.005) # Временная пауза для отображения графика
    globalEpochCounter+=1

    # Дополнение полной истории обучения
    totalHistoryLossTrain.extend(history.history['loss'])
    if 'val_loss' in history.history.keys():
        totalHistoryLossTest.extend(history.history['val_loss'])

Отобразим изменения значений потерь/ошибок по эпохам в виде графика. По виду этого графика вы можете оценить степень обученности НС

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения");
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка. Доп параметры color='black', linewidth=0.7
###plt.ylim ( (0, 0.03) ) # Область видимости для оси "Оценки потерь"

Вычислим ответы НС для тренировочных же данных (потому что мы ленивые)

In [ ]:
yNorm_pred = model.predict(xNorm_train)
stats(yNorm_train, yNorm_pred)
plot_difference(yNorm_train, yNorm_pred)

### Загрузка подготовленных данных

Структура csv-файла:
- Ячейки разделены ";"
- Дробная часть помечена "."

In [ ]:
dataset_file = "datasets/dataset_prepared.csv"
df = pd.read_csv(
    dataset_file,
    sep=';',
    decimal='.',
    header=0
)

df.head()

Выясним размеры датасета

In [ ]:
num_rows, num_cols = df.shape
print(f"Размеры набора данных: {num_rows} строк и {num_cols} столбцов\n")
df.info()

Целевой признак - Median Income: Медианный доход на семью в квартале[10тыс.$].

Коэфф. корреляции Пирсона позволит выбрать независимые переменные.

In [ ]:
target=['Median_Income']
corr_coeffs = df.corr(method='pearson')
corr_coeffs[target[0]].abs().sort_values(ascending=False)

В качестве независимых переменных выберем признаки с высоким абс. значением коэфф. корреляции, но при этом как можно более не связанные между собой. Кандидаты:
- *Median_House_Value* - Медианная цена дома в квартале [$]
- *Distance_to_coast*- расстояние до ближайшей точки побережья [м] (или *OcPrx_INLAND*)
- *Tot_Rooms* - Общее количество комнат в квартале
- *Median_Age* - Медианный возраст дома в квартале; меньше = новее [лет]

Дополнительно к ним добавим признак *Population*, в связке с которым при построении линейной модели проявилось свойство синергии - оценка R^2 была самой большой `R^2 = 0.465`

In [ ]:
features = ['Median_House_Value', 'Distance_to_coast', 'Tot_Rooms',  'Median_Age', 'Population']

Уберем все прочие признаки

In [ ]:
df = df[target +features]
df.head()

### Нормализация данных

Воспользуемся маштабированием через скейлер min-max. По [умолчанию](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html#:~:text=feature_rangetuple%20(min%2C%20max)%2C%20default%3D(0%2C%201)) скейлер выдает нормированный диапазон.

In [ ]:
minMaxScaler = MinMaxScaler()
df_scaled = pd.DataFrame (
  data    = minMaxScaler.fit_transform(df),
  columns = df.columns,
  index   = df.index
)
df_scaled.head()

Построим гистограммы распределения целевого признака

In [ ]:
def histograms(df, dfStd, feature_name):
    '''
    Построение гистограмм распределения признака до и после стандартизации
    '''
    plt.figure(figsize=(10,5))

    plt.subplot(121)
    plt.title('Распределение исходных значений')
    plt.xlabel(feature_name)
    plt.ylabel('Количество записей')
    plt.hist(df[feature_name])

    plt.subplot(122)
    plt.title('Распределение нормализованных значений')
    plt.xlabel(feature_name)
    plt.ylabel('Количество записей')
    plt.hist(dfStd[feature_name])

In [ ]:
histograms(df, df_scaled, 'Median_Income')

То же сделаем для независимых признаков

In [ ]:
for feat in features:
    histograms(df, df_scaled, feat)

### Формирование тренировочной и проверочной выборок

Зададим сид генератора случайных чисел. А также долю тестовой выборки.

In [ ]:
random_state = 42
test_size = 0.3

Разделим выборку на текстовую и тренировочную части.

In [ ]:
xNorm_train, xNorm_test, yNorm_train, yNorm_test = train_test_split(
    df_scaled[features],
    df_scaled[target],
    test_size=test_size,
    random_state=random_state,
    shuffle=True
)

yNorm_train = yNorm_train[target[0]]
yNorm_test  = yNorm_test[target[0]]

In [ ]:
print ("Кол-во элементов: \n  x_train: {}, y_train {} \n  x_test:  {}, y_test  {} \n  total x: {}, total y {} ".format  (
    len(xNorm_train), len(yNorm_train),
    len(xNorm_test),  len(yNorm_test),
    len(xNorm_train)+len(xNorm_test), len(yNorm_train)+len(xNorm_test),
))

### План работы

Построим модели НС, сосредотачиваясь на нескольких аспектах при постановке экспериментов:

- структуре сети: вход-выход, вход-слои-выход и множество слоев
- количество признаков
- количеству эпох тренировки
- величине групп, которыми обучается модель, *batch size*

### Нейронная сеть m1. Построение модели без скрытых слоев от ДВУХ "x" (2_вх->1_вых)

#### Структура

На НС со структурой 2вх->1вых накладываются требования:

- входной слой должен принимать 2 значения
- скрытые слои отсутствуют
- выходной слой вычисляет единственное значение и не имеет функцию активации

In [ ]:
totalHistoryLossTrain=[]
totalHistoryLossTest=[]

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input(shape=(2,)))
model.add(tf.keras.layers.Dense(units=1,  activation=None))

model.compile(
    loss      =fLoss,
    optimizer =fOptimizer,
    metrics   = fMetricList
)

print(model.summary())

#### Первичное и дополнительное обучение

Модель от 2х признаков обучим на *Median_House_Value* и *Tot_Rooms*

In [ ]:
features21=["Median_House_Value", "Tot_Rooms"]
xNorm_train_21=xNorm_train[features21]
xNorm_test_21=xNorm_test[features21]

Будем тренировать по 100 эпох.

In [ ]:
epochForTrain = 100

history = model.fit(
    xNorm_train_21,
    yNorm_train,
    validation_data=(xNorm_test_21,yNorm_test),
    epochs=epochForTrain,
    #batch_size=100, # внутри каждой эпохи, проводить обучение пакетами/порциями по batch_size строк
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

Посмотрим, как изменялась ошибка во времени.

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
###plt.ylim ( (0, 0.03) ) # Область видимости для оси "Оценки потерь"

И оценим объяснение тестовой выборки. 

In [ ]:
yNorm_pred = model.predict(xNorm_test_21).reshape(1, -1)[0] # потому что столбец
m1_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

Построим получаемую поверхность

In [ ]:
my3dplot(
    xNorm_train_21, yNorm_train,  True, # Обучающие x и y, Флаг отображения обучающих точек  (синие)
    xNorm_test_21,  yNorm_test,   True, # Проверочные x и y, Флаг отображения проверочных точек   (оранжевые)
    yNorm_pred,                True, # Вычисленные y, Флаг отображения вычисленной функции  (зелёные)
    x1Name= features21[0],  x2Name=features21[1], y_targetName=target[0],
    pointSize = 5,
    pointTransparency  = 0.2,
    meshTransparency   = 0.8
)

#### Получение весовых коэфф. w_i и bias для модели m1 (2_вх->1_вых)

Осмотрим параметры и значения весовых коэффициентов каждого отдельного слоя.

Обратим внимание на следующее:
- входной слой Input явно не выделен, но определяет входные параметры следующего за ним слоя;
- весовые коэффициенты при создании уже забиты случайными числами;
- слой с нелинейной функцией активации не имеет коэффициентов

In [ ]:
printModelWeights(model)

### Нейронная сеть m2. Построение модели со скрытыми слоями от ДВУХ "x" (2_вх-> ??? -> ??? ->1_вых)

Теперь построим модель с неколькими скрытыми слоями. Опять же, выбор количества слоев и фунцкий активации зависит от [приложения модели и ее архитектуры](https://towardsdatascience.com/activation-functions-in-neural-networks-how-to-choose-the-right-one-cb20414c04e5/#:~:text=The%20choice%20of%20the%20right,which%20function%20is%20most%20suitable.).

#### Модель с одним промежуточным слоем

Первую модель в эксперименте построим с единственным промежуточным слоем.

##### Структура

In [ ]:
totalHistoryLossTrain=[]
totalHistoryLossTest=[]

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input(shape=(2,)))
model.add(tf.keras.layers.Dense(units=50,  activation=tf.keras.activations.relu))
model.add(tf.keras.layers.Dense(units=1,  activation=None))

model.compile(
    loss      =fLoss,
    optimizer =fOptimizer,
    metrics   = fMetricList
)

print(model.summary())

##### Первичное и дополнительное обучение

In [ ]:
history = model.fit(
    xNorm_train_21,
    yNorm_train,
    validation_data=(xNorm_test_21,yNorm_test),
    epochs=epochForTrain
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
###plt.ylim ( (0, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test_21).reshape(1, -1)[0] # потому что столбец
m_1layer_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

##### Получение весовых коэфф. w_i и bias

Весовых коэффициентов стало много.

In [ ]:
printModelWeights(model)

Все также построим поверхность

In [ ]:
my3dplot(
    xNorm_train_21, yNorm_train,  True, # Обучающие x и y, Флаг отображения обучающих точек  (синие)
    xNorm_test_21,  yNorm_test,   True, # Проверочные x и y, Флаг отображения проверочных точек   (оранжевые)
    yNorm_pred,                True, # Вычисленные y, Флаг отображения вычисленной функции  (зелёные)
    x1Name= features21[0],  x2Name=features21[1], y_targetName=target[0],
    pointSize = 5,
    pointTransparency  = 0.2,
    meshTransparency   = 0.8
)

#### Модель с двумя промежуточными слоями

##### Структура

Введем еще один слой на больее число узлов

In [ ]:
totalHistoryLossTrain=[]
totalHistoryLossTest=[]

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input(shape=(2,)))
model.add(tf.keras.layers.Dense(units=200,  activation=tf.keras.activations.relu))
model.add(tf.keras.layers.Dense(units=50,  activation=tf.keras.activations.relu))
model.add(tf.keras.layers.Dense(units=1,  activation=None))

model.compile(
    loss      =fLoss,
    optimizer =fOptimizer,
    metrics   = fMetricList
)

print(model.summary())

##### Первичное и дополнительное обучение

In [ ]:
# epochForTrain = 40

history = model.fit(
    xNorm_train_21,
    yNorm_train,
    validation_data=(xNorm_test_21,yNorm_test),
    epochs=epochForTrain,
    #batch_size=100, # внутри каждой эпохи, проводить обучение пакетами/порциями по batch_size строк
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
###plt.ylim ( (0, 0.03) ) # Область видимости для оси "Оценки потерь"

Предскажем значения теста

In [ ]:
yNorm_pred = model.predict(xNorm_test_21).reshape(1, -1)[0] # потому что столбец
m_2layers_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

##### Получение весовых коэфф. w_i и bias для модели

In [ ]:
printModelWeights(model)

In [ ]:
my3dplot(
    xNorm_train_21, yNorm_train,  True, # Обучающие x и y, Флаг отображения обучающих точек  (синие)
    xNorm_test_21,  yNorm_test,   True, # Проверочные x и y, Флаг отображения проверочных точек   (оранжевые)
    yNorm_pred,                True, # Вычисленные y, Флаг отображения вычисленной функции  (зелёные)
    x1Name= features21[0],  x2Name=features21[1], y_targetName=target[0],
    pointSize = 5,
    pointTransparency  = 0.2,
    meshTransparency   = 0.8
)

#### Модель с четырьмя промежуточными слоями

##### Структура

Все четыре слоя - типа *Dense* с функцией *RELU*

In [ ]:
def initNewModel(size=5):
    totalHistoryLossTrain=[]
    totalHistoryLossTest=[]

    model = tf.keras.models.Sequential()
    model.add(tf.keras.layers.Input(shape=(size,)))

    model.add(tf.keras.layers.Dense(units=1000,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=500,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=200,  activation=tf.keras.activations.relu))
    model.add(tf.keras.layers.Dense(units=10,  activation=tf.keras.activations.relu))

    model.add(tf.keras.layers.Dense(units=1,  activation=None))

    model.compile(
        loss      =fLoss,
        optimizer =fOptimizer,
        metrics   = fMetricList
    )
    return model

model = initNewModel(2)
print(model.summary())

##### Первичное и дополнительное обучение

После 40 эпохи начинается раскалбас.

In [ ]:
# epochForTrain = 40

history = model.fit(
    xNorm_train_21,
    yNorm_train,
    validation_data=(xNorm_test_21,yNorm_test),
    epochs=epochForTrain,
    #batch_size=100, # внутри каждой эпохи, проводить обучение пакетами/порциями по batch_size строк
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
###plt.ylim ( (0, 0.03) ) # Область видимости для оси "Оценки потерь"

Предскажем значения теста

In [ ]:
yNorm_pred = model.predict(xNorm_test_21).reshape(1, -1)[0] # потому что столбец
m_4layers_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

##### Получение весовых коэфф. w_i и bias для модели

In [ ]:
printModelWeights(model)

In [ ]:
my3dplot(
    xNorm_train_21, yNorm_train,  True, # Обучающие x и y, Флаг отображения обучающих точек  (синие)
    xNorm_test_21,  yNorm_test,   True, # Проверочные x и y, Флаг отображения проверочных точек   (оранжевые)
    yNorm_pred,                True, # Вычисленные y, Флаг отображения вычисленной функции  (зелёные)
    x1Name= features21[0],  x2Name=features21[1], y_targetName=target[0],
    pointSize = 5,
    pointTransparency  = 0.2,
    meshTransparency   = 0.8
)

Выбор оптимального количества слоев достигается экспериментально. Нужно брать врассмотрение сложность и точность модели.

### Нейронная сеть m3. Построение модели без скрытых слоев от множества "x" (N_вх->1_вых))

#### Структура

Отличие - у входного слоя теперь 5 входных нейронов.

In [ ]:
totalHistoryLossTrain=[]
totalHistoryLossTest=[]

model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Input(shape=(5,)))
model.add(tf.keras.layers.Dense(units=1,  activation=None))

model.compile(
    loss      =fLoss,
    optimizer =fOptimizer,
    metrics   = fMetricList
)

#### Первичное и дополнительное обучение

In [ ]:
history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrain,
    #batch_size=100, # внутри каждой эпохи, проводить обучение пакетами/порциями по batch_size строк
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.018, 0.03) ) # Область видимости для оси "Оценки потерь"

Можем видеть как велика ошибка на начале обучения и как быстро она сводится на нет.

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m3_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

#### Получение весовых коэфф. w_i и bias

Чем меньше влияет признак тем меньше коэффициент.

In [ ]:
printModelWeights(model)

### Нейронная сеть m4. Построение модели со скрытыми слоями от множества "x" (N_вх-> ??? -> ??? ->1_вых)

#### Структура

Ранее заданные 4 промежуточных слоя с уменьшающимся количеством узлов.

In [ ]:
model = initNewModel()

#### Первичное и дополнительное обучение

In [ ]:
history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrain,
    #batch_size=100, # внутри каждой эпохи, проводить обучение пакетами/порциями по batch_size строк
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
# plt.ylim ( (0.0, 0.03) ) # Область видимости для оси "Оценки потерь"

Ошибка уменьшилась - точность возрасла.

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m4_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

### Нейронная сеть. Эксперименты с *batch size*

До этого модели тренировались с наборами по умолчанию, размер которых равен [32](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential#:~:text=per%20gradient%20update.-,If%20unspecified%2C%20batch_size%20will%20default%20to%2032,-.%20Do%20not%20specify
). Будем изменять это количество и смотреть на изменение оценок.

#### Размер 64

Исходная модель - 6 слоев.

In [ ]:
model = initNewModel()

Увеличим размер вдвое до 64.

In [ ]:
history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrain,
    batch_size=64
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

Оценка наилучшая из всех представленных.

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m64_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

#### Размер 128

In [ ]:
model = initNewModel()

In [ ]:
history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrain,
    batch_size=128
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m128_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

#### Размер 256

In [ ]:
model = initNewModel()

In [ ]:
history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrain,
    batch_size=256
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m256_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

#### Размер 1024

In [ ]:
model = initNewModel()

In [ ]:
history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrain,
    batch_size=1024
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m1024_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

### Нейронная сеть. Эксперимент с количеством эпох

#### Длительность в 80 эпох

Остановимся на размере 64. И будем изменять длительность обучения.

In [ ]:
optimal_batch=64
model = initNewModel()

In [ ]:
epochForTrainEx80 = 80

history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrainEx80,
    batch_size=optimal_batch
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m_80ep_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

#### Длительность в 160 эпох

In [ ]:
model = initNewModel()

In [ ]:
epochForTrainEx160 = 160

history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrainEx160,
    batch_size=optimal_batch
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m_160ep_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

#### Длительность в 20 эпох

In [ ]:
model = initNewModel()

In [ ]:
epochForTrainEx20 = 20

history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrainEx20,
    batch_size=optimal_batch
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m_20ep_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

#### Длительность в 500 эпох

Для этого эскперимента поднимем batch_size до 1024

In [ ]:
model = initNewModel()

In [ ]:
epochForTrainEx500 = 500

history = model.fit(
    xNorm_train,
    yNorm_train,
    validation_data=(xNorm_test,yNorm_test),
    epochs=epochForTrainEx500,
    batch_size=1024
)

totalHistoryLossTrain.extend(history.history['loss'])
if 'val_loss' in history.history.keys():
    totalHistoryLossTest.extend(history.history['val_loss'])

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(totalHistoryLossTrain, label='train', color = '#1f77b4')
if 'val_loss' in history.history.keys():
    plt.plot(totalHistoryLossTest, label='test', color = '#ff7f0e')
plt.title("История изменений выбраной оценки потерь (LossVal = mean_squared_error)")
plt.ylabel("Оценка потерь (mean_squared_error)"); plt.xlabel("Эпохи обучения")
plt.legend()
plt.grid(True, alpha=0.5)  # Сетка
plt.ylim ( (0.015, 0.03) ) # Область видимости для оси "Оценки потерь"

In [ ]:
yNorm_pred = model.predict(xNorm_test).reshape(1, -1)[0] # потому что столбец
m_500ep_stats = stats(yNorm_test, yNorm_pred)
plot_difference(yNorm_test, yNorm_pred)

Можем видеть как расходятся тренировочный и тестовый наборы на графиках ошибок

### Сводка по экспериментам

Эксперимент со слоями

In [ ]:
df_model_layers_results = pd.DataFrame(
    [ {'Признаки':features21, 'Число скрытых слоев': "1", "Эпохи обучения": epochForTrain, "R2":m_1layer_stats.R2, "RMSE":m_1layer_stats.RMSE},
      {'Признаки':features21, 'Число скрытых слоев': "2", "Эпохи обучения": epochForTrain, "R2":m_2layers_stats.R2, "RMSE":m_2layers_stats.RMSE},
      {'Признаки':features21, 'Число скрытых слоев': "4", "Эпохи обучения": epochForTrain, "R2":m_3layers_stats.R2, "RMSE":m_3layers_stats.RMSE},
    ]
)
df_model_layers_results

Экспермент со структурой

In [ ]:
df_model_results = pd.DataFrame(
    [ {'Признаки':features21, 'Структура': "2вх->1вых", "ЭпохОбучения": epochForTrain, "R2":m1_stats.R2, "RMSE":m1_stats.RMSE},
      {'Признаки':features21, 'Структура': "2вх-> ???  -> ??? ->1вых", "ЭпохОбучения":epochForTrain, "R2":m2_stats.R2, "RMSE":m2_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх->1вых", "ЭпохОбучения":epochForTrain, "R2":m3_stats.R2, "RMSE":m3_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх-> ??? -> ??? ->1вых", "ЭпохОбучения":epochForTrain, "R2":m4_stats.R2, "RMSE":m4_stats.RMSE},
    ]
)
df_model_results

Эксперимент с кусками выборки

In [ ]:
df_model_batch_results = pd.DataFrame(
    [ {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения": epochForTrain, "Размер":64, "R2":m_64batch_stats.R2, "RMSE":m_64batch_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения":epochForTrain, "Размер":128, "R2":m_128batch_stats.R2, "RMSE":m_128batch_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения":epochForTrain, "Размер":256, "R2":m_256batch_stats.R2, "RMSE":m_256batch_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения":epochForTrain, "Размер":1024, "R2":m_1024batch_stats.R2, "RMSE":m_1024batch_stats.RMSE},
    ]
)
df_model_batch_results

Эксперимент с эпохами

In [ ]:
df_model_epoch_results = pd.DataFrame(
    [ {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения": epochForTrain20, "R2":m_20ep_stats.R2, "RMSE":m_20ep_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения":epochForTrain80, "R2":m_80ep_stats.R2, "RMSE":m_80ep_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения":epochForTrain160, "R2":m_1600ep_stats.R2, "RMSE":m_160ep_stats.RMSE},
      {'Признаки':features, 'Структура': "5вх-> 4 ->1вых", "ЭпохОбучения":epochForTrain500, "R2":m_500ep_stats.R2, "RMSE":m_500ep_stats.RMSE},
    ]
)
df_model_epoch_results

### Оценка времени

In [ ]:
dfSpendTimeLab1 = pd.DataFrame.from_dict(
{
  "1": {"step": "Бизнес Анализ",                       "duration, min" :  5  },
  "2": {"step": "Анализ данных",                       "duration, min" :  5  },
  "3": {"step": "Подготовка данных",                   "duration, min" :  5  },
  "4": {"step": "Моделирование (Обуч.и подг.)",        "duration, min" :  120  },
  "5": {"step": "Моделирование (Оценка кач. моделей)", "duration, min" :  60  },
  "6": {"step": "Оценка решения + Внедрение",          "duration, min" :  60  },
}
, orient="index"
).sort_index(ascending=False)

# Построить столбчатую диаграмму
fig = plt.figure()
plt.barh(y = dfSpendTimeLab1["step"], width= dfSpendTimeLab1["duration, min"], )
plt.xlabel("Затраченное время, мин")

# Построить круговую диаграмму
fig = plt.figure()
plt.pie(x= dfSpendTimeLab1["duration, min"], labels=dfSpendTimeLab1["step"],  startangle = 90 )

plt.show()

---

## Выводы по работе

В ходе выполнения работы был опробован новый тип модели, основанной на стохастическом градиентном спуске. Стохастический характер проявляется, во-первых, в [требовании перемешивать входные данные](https://scikit-learn.ru/stable/modules/sgd.html#id12:~:text=%D0%A3%D0%B1%D0%B5%D0%B4%D0%B8%D1%82%D0%B5%D1%81%D1%8C%2C%20%D1%87%D1%82%D0%BE%20%D0%B2%D1%8B%20%D0%BF%D0%B5%D1%80%D0%B5%D0%BC%D0%B5%D1%88%D0%B8%D0%B2%D0%B0%D0%B5%D1%82%D0%B5%20(shuffle)%20%D0%B2%D0%B0%D1%88%D0%B8%20%D0%BE%D0%B1%D1%83%D1%87%D0%B0%D1%8E%D1%89%D0%B8%D0%B5%20%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D0%B5%20%D0%BF%D0%B5%D1%80%D0%B5%D0%B4%20%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5%D0%BC%20%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B8), во-вторых, от изменения зерна генератора случайных чисел изменялись и выходные характеристики модели. Третья особенность - необходимо приводить данные в вид, что дает среднее равное 0 и стандартное отклонение равное 1 - стандартизация; без нее значения модели были неадекватны и отличались на десятки порядков от ожидаемых.

Также был опробован прием кросс-валидации, позволяющий вычислять характеристики моделей на одном наборе данных с исключением некоторой части и тем самым получать более объективные оценки.